<a href="https://colab.research.google.com/github/EgemenYapucu/DataScienceExercises/blob/main/Quakecity.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install tensorflow

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.applications import ConvNeXtBase
from tensorflow.keras.layers import Dense, Flatten, Dropout
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt

height = 224
width = height
epochs = 6
batch_size = 32
num_classes = 3

train_dir = "C:/Users/EGEMEN/Desktop/QuakeCity/Labeled/ds/train"
test_dir = "C:/Users/EGEMEN/Desktop/QuakeCity/Labeled/ds/test"

datagen = ImageDataGenerator(
    rescale=1./255,  # Rescale pixel values to [0,1]
    shear_range=0.2,
    zoom_range=0.2,
)

train_generator = datagen.flow_from_directory(
    train_dir,
    target_size=(height, width),
    batch_size= batch_size,
    class_mode='categorical'
)

val_generator = datagen.flow_from_directory(
    test_dir,
    target_size= (height, width),
    batch_size= batch_size,
    class_mode='categorical',
    shuffle = False,
)

FileNotFoundError: [Errno 2] No such file or directory: 'C:/Users/EGEMEN/Desktop/QuakeCity/Labeled/ds/train'

In [ ]:
base_model = ConvNeXtBase(weights='imagenet', include_top=False, input_shape=(height, width, num_classes))

model = Sequential([
    base_model,
    Flatten(),
    Dense(1024,activation= 'relu'),
    Dropout(0.5),
    Dense(num_classes,activation = 'softmax')
    ])

model.compile(optimizer="adam",
              loss="categorical_crossentropy",
              metrics=["accuracy"])

In [ ]:
history = model.fit(
          train_generator,
          #steps_per_epoch = train_generator.samples // train_generator.batch_size,
          epochs = epochs,
          validation_data=val_generator,
          #validation_steps=val_generator.samples // val_generator.batch_size
          )

In [ ]:
val_generator.reset()
y_pred = model.predict(val_generator, steps=val_generator.samples // val_generator.batch_size + 1)
y_pred_classes = np.argmax(y_pred, axis=1)
y_true = val_generator.classes

precision = precision_score(y_true, y_pred_classes, average='weighted')
recall = recall_score(y_true, y_pred_classes, average='weighted')
f1 = f1_score(y_true, y_pred_classes, average='weighted')

results = model.evaluate(val_generator)
print(f"Validation Loss: {results[0]}")
print(f"Validation Accuracy: {results[1]}")
print(f"Validation Recall: {recall}")
print(f"Validation Precision: {precision}")
print(f"F1 Skor:{f1}")

In [ ]:
labels = list(val_generator.class_indices.keys())
y_true_labels = [labels[i] for i in y_true]
y_pred_labels = [labels[i] for i in y_pred_classes]
cm = confusion_matrix(y_true, y_pred_classes)

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(history.history['accuracy'], label='Train')
plt.plot(history.history['val_accuracy'], label='Validation')
plt.title('ConvNeXtBase Model Accuracy')
plt.ylabel('Accuracy')
plt.xlabel('Epochs')
plt.legend(loc='lower left')
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(history.history['loss'], label='Train')
plt.plot(history.history['val_loss'], label='Validation')
plt.title('ConvNeXtBase Model Loss')
plt.ylabel('Loss')
plt.xlabel('Epochs')
plt.legend(loc='lower left')
plt.show()

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
plt.xlabel('Predicted Label')
plt.ylabel('Actual Label')
plt.title('ConvNeXtBase Confusion Matrix')
plt.show